# Final Assignment: Analysing Gravitational Wave Event GW250114



### Submission Rules
For the final assignment of this course, you will perform data analysis on actual data. This submission must be uploaded on the Moodle page for this course under 'Assignment Uploads'. The only modules you will use for this assignment are NumPy, SciPy and Matplotlib.

## Overview

In this project, you will analyse real data from one of the most extreme events observable in the Universe: a binary black hole merger detected via gravitational waves. This event, [GW250114](https://journals.aps.org/prl/pdf/10.1103/kw5g-d732), was detected in January 2025 and is the loudest gravitational wave signal to date — and thus particularly attractive for testing General Relativity.

Gravitational waves are tiny distortions in spacetime produced by extremely energetic astrophysical systems. One of the most important sources is the merger of two compact objects (such as black holes). As these objects orbit each other they lose energy through gravitational-wave emission, resulting in their orbital separation decreasing and therefore the orbital frequency (and consequently gravitational wave frequency) increasing rapidly.

This produces a characteristic *chirp*: a signal whose frequency increases with time until a final merger.

<div style="text-align: center;">
  <img src="https://upload.wikimedia.org/wikipedia/commons/thumb/f/fd/Gw250114_data_and_reconstruction.svg/1920px-Gw250114_data_and_reconstruction.svg.png" width="1000">
</div>


### Data description

On the Moodle you will find the strain data for this event in the file `strain_data.txt`.  The file contains two columns: the GPS time array $t$ and the corresponding detector strain $h(t)$; this is a measure of the fractional change in length of the detector arms (in this case the Hanford detector).

The data has already been preprocessed. It has been *whitened*, meaning that the frequency-dependent detector noise has been normalised so that the noise spectrum is approximately flat. It has also been *bandpass filtered* to the frequency range 30-400 Hz, which removes low-frequency seismic noise and high-frequency quantum noise from the detector.

## Task 1

**Load the data and plot the detector strain $h(t)$.**

This is a simple time-domain visualisation of the signal. Include clearly labelled axes (with units) and ensure the plot is legible.


## Task 2

To visualise the chirp, we map the signal into a time-frequency representation. In gravitational-wave analyses, this is done using the *Q-transform*, a wavelet-based technique that is implemented efficiently using FFT methods and carefully optimised to handle large detector datasets. In this exercise, we use a simplified version of this idea, we directly compute how similar the signal is to a Gaussian-windowed complex sinusoid centred at each time $t_0$ and frequency $f_0$. For each $(t_0, f_0)$, we compute a complex projection:

$$
X(t_0, f_0) =
\sum_{t} h(t)\,
\exp\left[-\frac{(t-t_0)^2}{2\sigma^2}\right]
\exp(-i2\pi f_0 t)\,\Delta t
$$

We then define the time–frequency intensity as the normalised squared magnitude:

$$
P(t_0, f_0) =
\frac{|X(t_0, f_0)|^2}
{\sum_t
\exp\left[-\frac{(t-t_0)^2}{\sigma^2}\right]\Delta t}
$$

with

$$
\sigma = \frac{Q}{2\pi f_0}
$$

where $Q$ is the *quality factor*, which controls the trade-off between time and frequency resolution. This normalisation ensures that $P(t_0, f_0)$ is a dimensionless measure of signal intensity, independent of the width of the time–frequency window.

**Create a function `q_transform` that computes the time–frequency intensity map, and use it on your data.**

The function should take the inputs `t`, `h`, `freqs`, `times`, and `Q`. It should return a 2D NumPy array containing the intensity $P(t_0, f_0)$ evaluated on the grids defined by `times` and `freqs`.

Use a default value of `Q = 8`. The arrays `times` and `freqs` should each contain 200 evenly spaced points. `times` should span from `t[0]` to `t[-1]`, and `freqs` should span 20 Hz to 300 Hz.

Include a clear docstring describing the function, its inputs, and its output.

## Task 3

**Plot the time–frequency power map using `matplotlib.pyplot.contourf`.**


The horizontal axis should correspond to time (`times`) and the vertical axis to frequency (`freqs`). The power map should be plotted consistently with this convention. Include a colourbar showing the power, and ensure all axes are clearly labelled. You may find it necessary to transpose the power map when passing it to `contourf` depending on your implementation.

## Task 4

The frequency evolution of the signal is determined primarily by a combination of the masses in the system, known as the *chirp mass* $\mathcal{M}_c$,

$$
\mathcal{M}_c =
\frac{(m_1m_2)^{3/5}}{(m_1+m_2)^{1/5}}
$$

where $m_1$ and $m_2$ are the individual masses of the black holes.

From general relativity, the inspiral phase satisfies

$$
f^{-8/3}(t)=\frac{(8\pi)^{8/3}}{5}\left(\frac{G\mathcal{M}_c}{c^3}\right)^{5/3}(t_c - t),
$$

where $G$ is Newton's gravitational constant, $c$ is the speed of light, and $t_c$ is the coalescence time.

<br><br>

Since we have a time–frequency representation of the signal, we can estimate the chirp mass numerically.

<br><br>

**Obtain the frequency chirp track `f_track` by selecting, for each time step in `times`, the frequency in `freqs` at which the Q-transform intensity is maximised.**

## Task 5

**Plot $f_{\mathrm{track}}$ overlaid on the time–frequency intensity map.**

Do not worry if the chirp track is not perfectly smooth; this will be fitted over next.

## Task 6

We now estimate the chirp mass from the chirp track using a simple model fit. The inspiral relation can be rewritten using the variables

$$
x = f^{-8/3}, \qquad y = t_c - t,
$$

which gives a linear model of the form

$$
y = Ax
$$

where $A$ is related to the chirp mass $\mathcal{M}_c$ via

$$
A = \frac{5}{(8\pi)^{8/3}}
\left(\frac{G\mathcal{M}_c}{c^3}\right)^{-5/3}.
$$

**Use `scipy.optimize.curve_fit` to fit the model $y = Ax$ to your data and determine the best-fit value of $A$. From this value, compute the chirp mass $\mathcal{M}_c$ of the system in solar masses.**


When fitting, include only data points satisfying:

- frequency $30 \leq f \leq 300$ Hz,
- time $t \leq t_c$, where $t_c=1420878141.2241857$ s.


**Plot $y$ versus $x$ together with your fitted model to check that the fit is reasonable.**

The observed chirp mass was approximately $29\,M_{\odot}$. Don't worry if yours is larger - a deviation of up to 25% is acceptable due to noise, the simplified waveform model, and the approximate extraction of the chirp track. In realistic gravitational-wave analyses, more sophisticated waveform models and Bayesian inference methods are used.

## Task 7

**Finally, overlay your fitted chirp model on top of the Q-transform intensity map.**

Use your fitted parameters to reconstruct the model curve in the time–frequency plane (i.e. $f_{\mathrm{model}}(t)$), and plot it on the same axes as the Q-transform intensity map. Include a legend that reports the estimated chirp mass $\mathcal{M}_c$ to two decimal places.